In [1]:
import requests
from selenium import webdriver
import time
import pandas as pd
import random
import datetime

import glob 
import os 

In [2]:
path = ""
carpetas = ["2017/", "2018/", "2019/","2020/","2021/","2022/"]
file_list=[]

### Este script hace un pipeline en dos etapas:

### 1 - Construcción del universo de análisis (lista de CUITs de los proveedores que participaron en los procesos de compras obtenidos)
### 2 - Scraping profundo por proveedor, extrayendo múltiples dimensiones:
####    datos generales
####    estructura societaria
####    antecedentes
####    sanciones
####    documentación

### Etapa 1: construcción de la lista de cuits de proveedores a extraer

In [ ]:
for x in carpetas:
    for y in range (12):
        subcarpeta=path+x+str(y+1)+'/'
        file_list =file_list + [subcarpeta + f for f in os.listdir(subcarpeta) if f.startswith('pprocesos_proveedore')] 
        
file_list

In [4]:
#  crea la lista vacía que incluirá el contenido de cada archivo a convertir en pandas DF
csv_list = []

In [5]:
# 4. lee cada archivo (ordenado) en file_list, lo convierte en pandas DF y lo agrega a la lista csv_list
for file in sorted(file_list):
    csv_list.append(pd.read_csv(file,sep='|',header=None).assign(File_Name = os.path.basename(file)))
    
   

In [ ]:
csv_list

In [7]:
csv_merged = pd.concat(csv_list, ignore_index=True)

In [8]:
csv_merged

,0,1,2,3,4,5,6,7,8,9,10,File_Name
0,NaN,num_expediente,num_proceso,numero,estado,CUIT,nombre_proveedor,fecha_perfeccionamiento,monto,moneda,tipo_documento,pprocesos_proveedores.csv
1,0.0,EX-2017-00859614- -APN-DDMYA#SGP,23-0007-CDI17,23-1004-OC17,Perfeccionado,30678561165,NACION SEGUROS S.A.,30/1/2017,"158.161,99",Peso Argentino,Original,pprocesos_proveedores.csv
2,1.0,EX-2017-00859614- -APN-DDMYA#SGP,23-0007-CDI17,23-1054-OC17,Rescindido,30678561165,NACION SEGUROS S.A.,06/6/2017,"1.082,57",Peso Argentino,Ampliación,pprocesos_proveedores.csv
3,2.0,EX-2017-00859614- -APN-DDMYA#SGP,23-0007-CDI17,23-1060-OC17,Perfeccionado,30678561165,NACION SEGUROS S.A.,16/6/2017,"1.770,54",Peso Argentino,Ampliación,pprocesos_proveedores.csv
4,3.0,EX-2017-00859614- -APN-DDMYA#SGP,23-0007-CDI17,23-1104-OC17,Perfeccionado,30678561165,NACION SEGUROS S.A.,29/8/2017,"793,39",Peso Argentino,Ampliación,pprocesos_proveedores.csv
...,...,...,...,...,...,...,...,...,...,...,...,...
107903,1949.0,EX-2022-93434107- -APN-DACMYSG#HNDBS,97-0065-CDI22,97-0225-OC22,Perfeccionado,27252564778,ROMINA LAURA MICHALIK,27/10/2022,"95.928,00",Peso Argentino,Original,pprocesos_proveedores_adjudicado.csv
107904,1950.0,EX-2022-95867369- -APN-DACMYSG#HNDBS,97-0066-CDI22,97-0223-OC22,Perfeccionado,30522228210,Wiener Laboratorios SAIC,18/10/2022,"4.706.750,00",Peso Argentino,Original,pprocesos_proveedores_adjudicado.csv
107905,1951.0,EX-2022-92901770- -APN-DA#FMLCAV,98-0025-CDI22,98-0047-OC22,Perfeccionado,30500321586,Artes Gráficas S.A.,01/11/2022,"188.160,00",Peso Argentino,Original,pprocesos_proveedores_adjudicado.csv
107906,NaN,num_expediente,num_proceso,numero,estado,CUIT,nombre_proveedor,fecha_perfeccionamiento,monto,moneda,tipo_documento,pprocesos_proveedores_dejado_sin_efecto.csv


In [9]:
#arma la lista de CUITs sin los caracteres -
#lista_cuit =df['CUIT'].str.replace('-','')
lista_cuit=csv_merged[5]
lista_cuit=lista_cuit.drop_duplicates()
#lista_cuit=lista_cuit.remove('CUIT')
del lista_cuit[0]
lista_cuit

1           30678561165
6           20936959084
8           30650273652
10          30710967187
11          30708415487
              ...      
107824      30708855010
107833      30700503891
107834      30716441098
107835      27331126530
107878    30-70729747-2
Name: 5, Length: 12003, dtype: object

In [ ]:
#guarda la lista de cuits
lista_cuit.to_csv(path_or_buf=path+'proveedores\listaCuit.csv', sep='|', na_rep='', float_format=None, columns=None, header=False, mode='w', encoding='utf-8-sig', compression='infer', quoting=None, quotechar='"', line_terminator=None, chunksize=None, date_format=None, doublequote=True, escapechar=None, decimal='.', errors='strict', storage_options=None)


### Etapa 2: Scraping profundo por proveedor, extrayendo múltiples dimensiones:
#### datos generales
#### estructura societaria
#### antecedentes
#### sanciones
#### documentación

In [10]:
#lee la lista de cuits
#Lista a obtener
lista_cuit = pd.read_csv(path+"proveedores\listaCuit.csv.csv", sep='|')

#Para resto (fue necesario volver a extraer los datos de algunos proveedores que fallaron en el primer intento)
#lista_cuit = pd.read_csv("D:\comprar\proveedores\cuit_proveedores_resto2.csv", sep='|')

lista_cuit=lista_cuit[lista_cuit.columns[0]]
lista_cuit
lista_cuit=lista_cuit[lista_cuit.columns[0]]
lista_cuit

0          20047431841
1          20077944444
2          20078002175
3          20084980952
4          20100863163
            ...       
202    107510570RT0001
203          B83231878
204          B91224790
205       GSL000929KF5
206          J86153046
Name: 20043967755, Length: 207, dtype: object

In [21]:
search_query = 'https://comprar.gob.ar/PLIEGO/BuscarProveedorCiudadano.aspx'
#'https://www.indeed.com/q-data-scientist-jobs.html'
driver = webdriver.Chrome(executable_path='C:/chromedriver/chromedriver.exe')

In [5]:
#inicializa dataframes de cada dimensión a extraer
#print('inicializa dataframes')
datosProveedor_details = []
datosProveedor_info=[]

datosPF_details = []
datosPF_info=[]

clases_details = []
clases_doc_info=[]

apoderados_details = []
apoderados_info=[]

estado_doc_details = []
estado_doc_info=[]

socios_details = []
socios_info=[]

antecedentes_details = []
antecedentes_info=[]

sanciones_details = []
sanciones_info=[]


In [1]:
#grabar cada linea en un csv
def imprime_datos_proveedor():
    #print(contratos_details)
    datos=pd.DataFrame(datosProveedor_details, columns =['CUIT','Nombre', 'FechaPreinscripcion', 'Estado', 'TelefonicoContacto', 'TelefonicoAlternativo','DomEspecialElectronico','Constitucion', 'TipoSocietario', 'NumeroEnte', 'DomicilioLegal', 'DomicilioEspecial', 'CorreoInstitucional','RegistroPublicoComercio','InspeccionGeneralJusticia'])
    datos.to_csv(path_or_buf='D:\comprar\proveedores\datos_proveedor.csv', sep='|', na_rep='', float_format=None, columns=None, header=True, mode='w', encoding='utf-8-sig', compression='infer', quoting=None, quotechar='"', line_terminator=None, chunksize=None, date_format=None, doublequote=True, escapechar=None, decimal='.', errors='strict', storage_options=None)
  
    datos=pd.DataFrame(clases_details, columns =['CUIT', 'Nro_clases', 'codigo_clase','descripcion_clase','codigo_rubro','descripcion_rubro'])
    datos.to_csv(path_or_buf='D:\comprar\proveedores\datos_clases.csv', sep='|', na_rep='', float_format=None, columns=None, header=True, mode='w', encoding='utf-8-sig', compression='infer', quoting=None, quotechar='"', line_terminator=None, chunksize=None, date_format=None, doublequote=True, escapechar=None, decimal='.', errors='strict', storage_options=None)
    
    datos=pd.DataFrame(apoderados_details, columns =['CUIT','nombre_apellido','cuit_apoderado','dni','tipo_representacion','monto_lim_oferta','adm_legitimado'])
    datos.to_csv(path_or_buf='D:\comprar\proveedores\datos_apoderados.csv', sep='|', na_rep='', float_format=None, columns=None, header=True, mode='w', encoding='utf-8-sig', compression='infer', quoting=None, quotechar='"', line_terminator=None, chunksize=None, date_format=None, doublequote=True, escapechar=None, decimal='.', errors='strict', storage_options=None)
    
    datos=pd.DataFrame(estado_doc_details, columns =['CUIT','descripcion','fecha'])
    datos.to_csv(path_or_buf='D:\comprar\proveedores\datos_estado_documentacion.csv', sep='|', na_rep='', float_format=None, columns=None, header=True, mode='w', encoding='utf-8-sig', compression='infer', quoting=None, quotechar='"', line_terminator=None, chunksize=None, date_format=None, doublequote=True, escapechar=None, decimal='.', errors='strict', storage_options=None)
    
    datos=pd.DataFrame(antecedentes_details, columns =['CUIT','fecha_publicacion','tipo_antedecedente','organismo_emisor','causa','encuadre_legal','observaciones'])
    datos.to_csv(path_or_buf='D:\comprar\proveedores\datos_antecedentes.csv', sep='|', na_rep='', float_format=None, columns=None, header=True, mode='w', encoding='utf-8-sig', compression='infer', quoting=None, quotechar='"', line_terminator=None, chunksize=None, date_format=None, doublequote=True, escapechar=None, decimal='.', errors='strict', storage_options=None)
    
    datos=pd.DataFrame(sanciones_details, columns =['CUIT','tipo_sancion','estado','plazo','fecha_inicio','fecha_fin','encuadre_legal','observaciones'])
    datos.to_csv(path_or_buf='D:\comprar\proveedores\datos_sanciones.csv', sep='|', na_rep='', float_format=None, columns=None, header=True, mode='w', encoding='utf-8-sig', compression='infer', quoting=None, quotechar='"', line_terminator=None, chunksize=None, date_format=None, doublequote=True, escapechar=None, decimal='.', errors='strict', storage_options=None)


In [7]:
#grabar cada linea en un csv
def imprime_datos_pf():
    #print(contratos_details)
    datos=pd.DataFrame(datosPF_details, columns =['CUIT','Nombre','Apellido','tipoDoc','nroDoc','nacionalidad','email', 'emailAlternativo','estadoCivil'])
    datos.to_csv(path_or_buf='D:\comprar\proveedores\datos_persona_fisica.csv', sep='|', na_rep='', float_format=None, columns=None, header=True, mode='w', encoding='utf-8-sig', compression='infer', quoting=None, quotechar='"', line_terminator=None, chunksize=None, date_format=None, doublequote=True, escapechar=None, decimal='.', errors='strict', storage_options=None)
  
    


In [8]:
#grabar cada linea en un csv
def imprime_datos_sociedades():
    #print(contratos_details)
    datos=pd.DataFrame(socios_details, columns =['CUIT','nombre_apellido','cuit_socio','doc_socio','cargo'])
    datos.to_csv(path_or_buf='D:\comprar\proveedores\datos_socios.csv', sep='|', na_rep='', float_format=None, columns=None, header=True, mode='w', encoding='utf-8-sig', compression='infer', quoting=None, quotechar='"', line_terminator=None, chunksize=None, date_format=None, doublequote=True, escapechar=None, decimal='.', errors='strict', storage_options=None)
  
    



In [ ]:
#arma la lista de CUITs sin los caracteres -
#lista_cuit =df['CUIT'].str.replace('-','')
lista_cuit =df['CUIT']
lista_cuit=lista_cuit.drop_duplicates()
lista_cuit

In [9]:
#grabar cada linea en un csv
def guarda_rubros_info():
    #print(contratos_details)
    datos=pd.DataFrame(rubros_info, columns =['nroDoc','codigo_clase','descripcion_clase','codigo_rubro','descripcion_rubro'])
    #print(datos)
    datos.to_csv(path_or_buf='D:\Vanina\COMP_AR\proveedores\rubro_proveedores.csv', sep='|', na_rep='', float_format=None, columns=None, header=True, mode='w', encoding='utf-8-sig', compression='infer', quoting=None, quotechar='"', line_terminator=None, chunksize=None, date_format=None, doublequote=True, escapechar=None, decimal='.', errors='strict', storage_options=None)


In [10]:
#grabar cada linea en un csv
def imprime_datos_pag(fecha_desde, fecha_hasta):
    #print(contratos_details)
    datos=pd.DataFrame(contratos_details, columns =['num_expediente', 'num_proceso', 'etapa', 'modalidad', 'moneda', 'encuadre_legal', 'cotizacion', 'tipo_proceso', 'lugar_recepcion', 'plazo_oferta', 'requiere_pago','apartado','etapa_lic','etapa_autorizacion_pliego','etapa_autorizacion_llamado', 'etapa_acto_apertura', 'genera_recursos', 'financiamiento_externo','acepta_prorroga','tipo_de_bienes','genera_recursos_mediante','cro_fecha_publicacion','cro_fecha_inicio_consultas','cro_fecha_final_consultas','cro_cant_dias_publicar','cro_fecha_inicio_recepcion_documentos','cro_fecha_fin_recepcion_documentos','cro_fecha_acto_apertura','inicio_contrato','duracion_contrato','proveedores_participantes','ofertas_confirmadas'])
    #print(datos)
    datos.to_csv(path_or_buf='D:\Vanina\COMP_AR\procesos\procesos_'&fecha_desde&'_'&fecha_hasta&'.csv', sep='|', na_rep='', float_format=None, columns=None, header=True, mode='w', encoding='utf-8-sig', compression='infer', quoting=None, quotechar='"', line_terminator=None, chunksize=None, date_format=None, doublequote=True, escapechar=None, decimal='.', errors='strict', storage_options=None)
    #datos.to_csv(path_or_buf='D:\Vanina\COMP_AR\procesos\procesos.csv', sep=';', na_rep='', header=True, encoding='utf-8-sig')



In [11]:
def obtiene_datos_proveedor():
    
    #DATOS DEL PROVEEDOR
    print('entra obtiene datos proveedor')
    try:
        Nombre=driver.find_element_by_id('ctl00_CPH1_UCVerCertificadoEstadoRegistralCiudadano_lblNombreFantasia').text
    except Exception:
        Nombre=driver.find_element_by_id('ctl00_CPH1_UCVerCertificadoEstadoRegistralCiudadano_lblRazonSocial').text
    print(Nombre)
    CUIT=driver.find_element_by_id("ctl00_CPH1_UCVerCertificadoEstadoRegistralCiudadano_lblNumeroCUIT").text.replace('-', '')
    print(CUIT)
    FechaPreinscripcion=driver.find_element_by_id('ctl00_CPH1_UCVerCertificadoEstadoRegistralCiudadano_lblFechaPreinscripcion').text
    #print(FechaPreinscripcion)
    Estado=driver.find_element_by_id('ctl00_CPH1_UCVerCertificadoEstadoRegistralCiudadano_lblEstado').text
    #print(Estado)
    try:
        TelefonicoContacto=driver.find_element_by_id('ctl00_CPH1_UCVerCertificadoEstadoRegistralCiudadano_lblTelefonicoContacto').text
    except Exception:
        TelefonicoContacto = ''
    #print(TelefonicoContacto)
    try:
        TelefonicoAlternativo=''#driver.find_element_by_id('ctl00_CPH1_UCVerCertificadoEstadoRegistralCiudadano_lblTelefonicoContacto').text
    except Exception:
        TelefonicoAlternativo = ''
    #print(TelefonicoAlternativo)
    try:
        DomEspecialElectronico=''#driver.find_element_by_id('ctl00_CPH1_UCVerCertificadoEstadoRegistralCiudadano_lblTelefonicoContacto').text
    except Exception:
        DomEspecialElectronico = ''
    #print(DomEspecialElectronico)
    try:
        Constitucion =driver.find_element_by_id('ctl00_CPH1_UCVerCertificadoEstadoRegistralCiudadano_lblConstitucion').text
    except Exception:
        Constitucion = ''
    #print(Constitucion)
    try:
        TipoSocietario=driver.find_element_by_id('ctl00_CPH1_UCVerCertificadoEstadoRegistralCiudadano_lblTipoSocietario').text
    except Exception:
        TipoSocietario = ''
    #print(TipoSocietario)
    try:
        NumeroEnte=driver.find_element_by_id('ctl00_CPH1_UCVerCertificadoEstadoRegistralCiudadano_lblNumeroEnte').text
    except Exception:
        NumeroEnte = ''
    #print(NumeroEnte)
    try:
        DomicilioLegal=driver.find_element_by_id('ctl00_CPH1_UCVerCertificadoEstadoRegistralCiudadano_lblDomicilioLegal').text
    except Exception:
        DomicilioLegal = ''
    #print(DomicilioLegal)
    try:
        DomicilioEspecial=driver.find_element_by_id('ctl00_CPH1_UCVerCertificadoEstadoRegistralCiudadano_lblDomicilioConstituido').text
    except Exception:
        DomicilioEspecial = ''
    #print(DomicilioEspecial) 
    try:
        CorreoInstitucional=driver.find_element_by_id('ctl00_CPH1_UCVerCertificadoEstadoRegistralCiudadano_lblCorreoInstitucional').text
    except Exception:
        CorreoInstitucional = ''
    #print(CorreoInstitucional) 
    
    #Números de inscripción
    try:
        RegistroPublicoComercio=driver.find_element_by_id('ctl00_CPH1_UCVerCertificadoEstadoRegistralCiudadano_lblRegistroPublicoComercio').text
    except Exception:
        RegistroPublicoComercio = ''
        print('error RegistroPublicoComercio')
        
    try:
        InspeccionGeneralJusticia=driver.find_element_by_id('ctl00_CPH1_UCVerCertificadoEstadoRegistralCiudadano_lblInspeccionGeneralJusticia').text
    except Exception:
        InspeccionGeneralJusticia = ''
        print('error InspeccionGeneralJusticia')
    
    
    datosProveedor_info = [ CUIT,Nombre, FechaPreinscripcion, Estado, TelefonicoContacto, TelefonicoAlternativo,DomEspecialElectronico,Constitucion, TipoSocietario, NumeroEnte, DomicilioLegal, DomicilioEspecial, CorreoInstitucional,RegistroPublicoComercio,InspeccionGeneralJusticia]
    datosProveedor_details.append(datosProveedor_info)  
    
    print('sale obtiene datos proveedor')



In [12]:
#Datos de la persona física
def obtener_datos_persona_fisica():
   print('entra PF')
   CUIT=driver.find_element_by_id("ctl00_CPH1_UCVerCertificadoEstadoRegistralCiudadano_lblNumeroCUIT").text.replace('-', '')
   print(CUIT)
   try:
       Nombre=driver.find_element_by_id('ctl00_CPH1_UCVerCertificadoEstadoRegistralCiudadano_lblNombrePersonaFisica').text
   except Exception:
       Nombre = ''
   #print(Nombre) 
   try:
       Apellido=driver.find_element_by_id('ctl00_CPH1_UCVerCertificadoEstadoRegistralCiudadano_lblApellidoPersonaFisica').text
   except Exception:
       Apellido = ''
   #print(Apellido) 
   try:
       tipoDoc=driver.find_element_by_id('ctl00_CPH1_UCVerCertificadoEstadoRegistralCiudadano_lblTipoDocumentoPersonaFisica').text
   except Exception:
       tipoDoc = ''
   #print(tipoDoc) 
   try:
       nroDoc=driver.find_element_by_id('ctl00_CPH1_UCVerCertificadoEstadoRegistralCiudadano_lblNumeroDocumentoPersonaFisica').text
   except Exception:
       nroDoc = ''
   #print(nroDoc) 
   try:
       nacionalidad=driver.find_element_by_id('ctl00_CPH1_UCVerCertificadoEstadoRegistralCiudadano_lblNacionalidadPersonaFisica').text
   except Exception:
       nacionalidad = ''
   #print(nacionalidad) 
   try:
       email=driver.find_element_by_id('ctl00_CPH1_UCVerCertificadoEstadoRegistralCiudadano_lblEmailPersonaFisica').text
   except Exception:
       email = ''
   #print(email)
   try:
       emailAlternativo=''#driver.find_element_by_id('ctl00_CPH1_UCVerCertificadoEstadoRegistralCiudadano_lblEmailPersonaFisica').text
   except Exception:
       emailAlternativo = ''
   #print(emailAlternativo)
   try:
       estadoCivil=driver.find_element_by_id('ctl00_CPH1_UCVerCertificadoEstadoRegistralCiudadano_lblEstadoCivilPersonaFisica').text
   except Exception:
       estadoCivil = ''
   #print(estadoCivil)    
   
   datosPF_info = [CUIT,Nombre,Apellido,tipoDoc,nroDoc,nacionalidad,email, emailAlternativo,estadoCivil]
   datosPF_details.append(datosPF_info)  
   
   print('sale apoderado')
       


In [13]:
#Clases inscriptas - Obtiene los rubros
def obtiene_clases():
    print('entra obtiene datos clases')
    CUIT=driver.find_element_by_id("ctl00_CPH1_UCVerCertificadoEstadoRegistralCiudadano_lblNumeroCUIT").text.replace('-', '')
    
    tabla_rubros = driver.find_element_by_id('ctl00_CPH1_UCVerCertificadoEstadoRegistralCiudadano_gvClasesInscriptas')
    rows = tabla_rubros.find_elements_by_xpath('//*[@id="ctl00_CPH1_UCVerCertificadoEstadoRegistralCiudadano_gvClasesInscriptas"]/tbody/tr')
    number_of_rows = len(rows)
    print('nro de rows a iterar dentro de la tabla Rubros: '+str(number_of_rows))
    for row in rows:
        # Get the columns(all the column 2)
        cols = row.find_elements_by_tag_name("td")
        number_of_cols = len(cols)
        #print(number_of_cols)
        if number_of_cols > 0:
            try:
                codigo_clase=cols[0].text 
            except Exception:
                codigo_clase=''
                print('error codigo clase')
            #print(codigo_clase)
            try:
                descripcion_clase= cols[1].text.replace(';',' -').replace("\n", " ")
            except Exception:
                descripcion_clase=''
                print('error descripcion_clase')
            #print(descripcion_clase)    
            try:
                codigo_rubro = cols[2].text
            except Exception:
                codigo_rubro=''
                print('error codigo_rubro')
            #print(codigo_rubro)  
            try:
                descripcion_rubro = cols[3].text.replace(';',' -').replace("\n", " ")
            except Exception:
                descripcion_rubro=''
                print('error descripcion_rubro')
            #print(descripcion_rubro)
            clases_info = [CUIT, number_of_rows -1, codigo_clase,descripcion_clase,codigo_rubro,descripcion_rubro]
            clases_details.append(clases_info) 
            if(number_of_rows > 50):
                time.sleep(random.uniform(0.01,0.02)) 
            
    print('sale obtiene datos clases')



In [ ]:
datos=pd.DataFrame(clases_details, columns =['CUIT', 'codigo_clase','descripcion_clase','codigo_rubro','descripcion_rubro'])
datos.to_csv(path_or_buf='D:\Vanina\COMP_AR\proveedores\scraping\datos_clases_caso_remanente.csv', sep='|', na_rep='', float_format=None, columns=None, header=True, mode='w', encoding='utf-8-sig', compression='infer', quoting=None, quotechar='"', line_terminator=None, chunksize=None, date_format=None, doublequote=True, escapechar=None, decimal='.', errors='strict', storage_options=None)
    


In [14]:
#Representante Legal / Apoderado
def obtiene_apoderado():    
    print('entra apoderado')
    CUIT=driver.find_element_by_id("ctl00_CPH1_UCVerCertificadoEstadoRegistralCiudadano_lblNumeroCUIT").text.replace('-', '')
        
    tabla_apoderado = driver.find_element_by_id('ctl00_CPH1_UCVerCertificadoEstadoRegistralCiudadano_gvAdministradoresLegitimados')
    rows = tabla_apoderado.find_elements_by_xpath('//*[@id="ctl00_CPH1_UCVerCertificadoEstadoRegistralCiudadano_gvAdministradoresLegitimados"]/tbody/tr')
    number_of_rows = len(rows)
    print('nro de rows a iterar dentro de la tabla Apoderado: '+str(number_of_rows))
    for row in rows:
        # Get the columns(all the column 2)
        cols = row.find_elements_by_tag_name("td")
        number_of_cols = len(cols)
        #print(number_of_cols)
        if number_of_cols > 0:
            try:
                nombre_apellido=cols[0].text.replace(';',' -').replace("\n", " ") 
            except Exception:
                nombre_apellido=''
                print('error nombre_apellido')
            #print(nombre_apellido)
            try:
                cuit_apoderado= cols[1].text
            except Exception:
                cuit_apoderado=''
                print('error cuit_apoderado')
            #print(cuit)    
            try:
                dni = cols[2].text
            except Exception:
                dni=''
                print('error dni')
            #print(dni)  
            try:
                tipo_representacion = cols[3].text.replace(';',' -').replace("\n", " ")
            except Exception:
                tipo_representacion=''
                print('error tipo_representacion')
            #print(tipo_representacion)
            try:
                monto_lim_oferta = cols[4].text
            except Exception:
                monto_lim_oferta=''
                print('error monto_lim_oferta')
            #print(monto_lim_oferta)
            try:
                adm_legitimado = cols[5].text
            except Exception:
                adm_legitimado=''
                print('error adm_legitimado')
            #print(adm_legitimado)
            
            apoderados_info = [CUIT,nombre_apellido,cuit_apoderado,dni,tipo_representacion,monto_lim_oferta,adm_legitimado]
            apoderados_details.append(apoderados_info)    
        
            
    print('sale apoderado')


In [15]:
#Estado de la documentación
def obtiene_estado_doc():  
    
    CUIT=driver.find_element_by_id("ctl00_CPH1_UCVerCertificadoEstadoRegistralCiudadano_lblNumeroCUIT").text.replace('-', '')
       
    tabla_doc = driver.find_element_by_id('ctl00_CPH1_UCVerCertificadoEstadoRegistralCiudadano_gvDocumentos')
    rows = tabla_doc.find_elements_by_xpath('//*[@id="ctl00_CPH1_UCVerCertificadoEstadoRegistralCiudadano_gvDocumentos"]/tbody/tr')
    number_of_rows = len(rows)
    print('nro de rows a iterar dentro de la tabla Documentacion: '+str(number_of_rows))
    for row in rows:
        # Get the columns(all the column 2)
        cols = row.find_elements_by_tag_name("td")
        number_of_cols = len(cols)
        #print(number_of_cols)
        if number_of_cols > 0:
            try:
                descripcion=cols[0].text.replace(';',' -').replace("\n", " ") 
            except Exception:
                descripcion=''
                print('error descripcion')
            #print(descripcion)
            try:
                fecha= cols[1].text
            except Exception:
                fecha=''
                print('error fecha')
            #print(fecha)    
            estado_doc_info = [CUIT,descripcion,fecha]
            estado_doc_details.append(estado_doc_info)    
        if(number_of_rows > 50):
            time.sleep(random.uniform(0.1,0.2)) 
            


In [16]:
def obtiene_datos_socios():  
    print('entra socios')
    CUIT=driver.find_element_by_id("ctl00_CPH1_UCVerCertificadoEstadoRegistralCiudadano_lblNumeroCUIT").text.replace('-', '')
       
    tabla_socios = driver.find_element_by_id('ctl00_CPH1_UCVerCertificadoEstadoRegistralCiudadano_gvSocios')
    rows = tabla_socios.find_elements_by_xpath('//*[@id="ctl00_CPH1_UCVerCertificadoEstadoRegistralCiudadano_gvSocios"]/tbody/tr')
    number_of_rows = len(rows)
    print('nro de rows a iterar dentro de la tabla socios: '+str(number_of_rows))
    for row in rows:
        # Get the columns(all the column 2)
        cols = row.find_elements_by_tag_name("td")
        number_of_cols = len(cols)
        #print(number_of_cols)
        if number_of_cols > 0:
            try:
                nombre_apellido=cols[0].text 
            except Exception:
                nombre_apellido=''
                print( 'error nombre_apellido')
            #print(descripcion)
            try:
                cuit_socio= cols[1].text.replace('-', '')
            except Exception:
                cuit_socio=''
                print('error cuit_socio')
            try:
                doc_socio= cols[2].text
            except Exception:
                doc_socio=''
                print('error doc_socio')
            try:
                cargo= cols[3].text
            except Exception:
                cargo=''
                print('error cargo')
            #print(fecha)    
            socios_info = [CUIT,nombre_apellido,cuit_socio,doc_socio,cargo]
            socios_details.append(socios_info)    
       
    print('sale socios')
            


In [17]:
def obtiene_datos_antecedentes():  
    print('entra antecedentes')
    CUIT=driver.find_element_by_id("ctl00_CPH1_UCVerCertificadoEstadoRegistralCiudadano_lblNumeroCUIT").text.replace('-', '')
       
    tabla_antecedentes = driver.find_element_by_id('ctl00_CPH1_UCVerCertificadoEstadoRegistralCiudadano_gvAntecedentes')
    rows = tabla_antecedentes.find_elements_by_xpath('//*[@id="ctl00_CPH1_UCVerCertificadoEstadoRegistralCiudadano_gvAntecedentes"]/tbody/tr')
    number_of_rows = len(rows)
    print('nro de rows a iterar dentro de la tabla antedecentes: '+str(number_of_rows))
    for row in rows:
        # Get the columns(all the column 2)
        cols = row.find_elements_by_tag_name("td")
        number_of_cols = len(cols)
        #print(number_of_cols)
        if number_of_cols > 0:
            try:
                fecha_publicacion=cols[0].text 
            except Exception:
                fecha_publicacion=''
                print( 'error fecha_publicacion')
            #print(descripcion)
            try:
                tipo_antedecedente= cols[1].text.replace(',', '')
            except Exception:
                tipo_antedecedente=''
                print('error tipo_antedecedente')
            try:
                organismo_emisor= cols[2].text.replace(',', '').replace("\n", " ")
            except Exception:
                organismo_emisor=''
                print('error organismo_emisor')
            try:
                causa= cols[3].text.replace(',', '').replace("\n", " ")
            except Exception:
                causa=''
                print('error causa').text.replace(',', '')
            try:
                encuadre_legal= cols[4].text.replace(',', '').replace("\n", " ")
            except Exception:
                encuadre_legal=''
                print('error encuadre_legal').text.replace(',', '')
            try:
                observaciones= cols[5].text.replace(',', '').replace("\n", " ")
            except Exception:
                observaciones=''
                print('error observaciones').text.replace(',', '').replace("\n", " ")
            #print(fecha)    
            antecedentes_info = [CUIT,fecha_publicacion,tipo_antedecedente,organismo_emisor,causa,encuadre_legal,observaciones]
            antecedentes_details.append(antecedentes_info)    
       
    print('sale antecedentes')
            


In [18]:
def obtiene_datos_sanciones():  
    print('entra sanciones')
    CUIT=driver.find_element_by_id("ctl00_CPH1_UCVerCertificadoEstadoRegistralCiudadano_lblNumeroCUIT").text.replace('-', '')
       
    tabla_antecedentes = driver.find_element_by_id('ctl00_CPH1_UCVerCertificadoEstadoRegistralCiudadano_gridSanciones')
    rows = tabla_antecedentes.find_elements_by_xpath('//*[@id="ctl00_CPH1_UCVerCertificadoEstadoRegistralCiudadano_gridSanciones"]/tbody/tr')
    number_of_rows = len(rows)
    print('nro de rows a iterar dentro de la tabla sanciones: '+str(number_of_rows))
    for row in rows:
        # Get the columns(all the column 2)
        cols = row.find_elements_by_tag_name("td")
        number_of_cols = len(cols)
        #print(number_of_cols)
        if number_of_cols > 0:
            try:
                tipo_sancion=cols[0].text 
            except Exception:
                tipo_sancion=''
                print( 'error tipo_sancion')
            #print(descripcion)
            try:
                estado= cols[1].text
            except Exception:
                estado=''
                print('error estado')
            try:
                plazo= cols[2].text
            except Exception:
                plazo=''
                print('error plazo')
            try:
                fecha_inicio= cols[3].text.replace(',', '').replace("\n", " ")
            except Exception:
                fecha_inicio=''
                print('error fecha_inicio').text.replace(',', '')
            try:
                fecha_fin= cols[4].text.replace(',', '').replace("\n", " ")
            except Exception:
                fecha_fin=''
                print('error fecha_fin').text
            try:
                encuadre_legal= cols[5].text.replace(',', '').replace("\n", " ")
            except Exception:
                encuadre_legal=''
                print('error encuadre_legal').text.replace(',', '').replace("\n", " ")
            try:
                observaciones= cols[6].text.replace(',', '').replace("\n", " ")
            except Exception:
                observaciones=''
                print('error observaciones').text.replace(',', '').replace("\n", " ")
            #print(fecha)    
            sanciones_info = [CUIT,tipo_sancion,estado,plazo,fecha_inicio,fecha_fin,encuadre_legal,observaciones]
            sanciones_details.append(sanciones_info)    
       
    print('sale sanciones')


In [22]:
#itera por páginas

for cuit in lista_cuit:
    print('-------------------------------------------------------------------------')
    print(cuit)
    #carga la página y le pone los parámetros
    driver.get(search_query)
    time.sleep(random.uniform(3.0,5.0))   
    driver.find_element_by_id('ctl00_CPH1_UCBuscarProveedor_txtNumeroCUITCUIL').clear()
    input_cuit_proveedor = driver.find_element_by_id('ctl00_CPH1_UCBuscarProveedor_txtNumeroCUITCUIL')
    #print(input_cuit_proveedor)
    input_cuit_proveedor.send_keys(cuit)
    btn_cuit_proveedor=  driver.find_element_by_id('ctl00_CPH1_UCBuscarProveedor_btnBusquedaRapida') 
    btn_cuit_proveedor.click()
    time.sleep(random.uniform(3.0,5.0))   
    
  
    #tabla_doc = driver.find_element_by_id('ctl00_CPH1_UCVerCertificadoEstadoRegistralCiudadano_gvDocumentos')
    #rows = tabla_doc.find_elements_by_xpath('//*[@id="ctl00_CPH1_UCVerCertificadoEstadoRegistralCiudadano_gvDocumentos"]/tbody/tr')
    #number_of_rows = len(rows)
    
    #tabla de proveedores
    try:
        tabla_principal = driver.find_element_by_id('ctl00_CPH1_UCBuscarProveedor_gvResultados')
        rows = tabla_principal.find_elements_by_xpath('//*[@id="ctl00_CPH1_UCBuscarProveedor_gvResultados"]/tbody/tr[2]')#//*[@id="ctl00_CPH1_UCBuscarProveedor_gvResultados"]/tbody/tr
        number_of_rows = len(rows)
        print('nro de rows a iterar dentro de la tabla: '+str(number_of_rows))
        #recorrer toda la tabla de proveedores
        for row in rows:
            link_pagina = driver.find_element_by_id('ctl00_CPH1_UCBuscarProveedor_gvResultados_ctl02_lnkNumeroCUITCUIL')
            link_pagina.click() 
            time.sleep(random.uniform(5.0,3.0))     
            
            #extrae datos comunes a todos los tipos de proveedores
            obtiene_datos_proveedor()
            time.sleep(random.uniform(1.0,2.0))
            try:
                obtiene_clases()
            except Exception:
                print('****error en clases ****')            
            time.sleep(random.uniform(4.0,5.0))
            try:
                obtiene_apoderado()
            except Exception:
                print('****error en apoderado ****')
            time.sleep(random.uniform(1.0,2.0))
            try:
                obtiene_estado_doc()
            except Exception:
                print('****error en documentos ****')
            time.sleep(random.uniform(1.0,2.0))
            try:
                obtiene_datos_antecedentes()                  
            except Exception:
                print('no tiene antecedentes')   
            time.sleep(random.uniform(1.0,2.0))
            try:
                obtiene_datos_sanciones()              
            except Exception:
                print('no tiene sanciones')
            time.sleep(random.uniform(1.0,2.0))
            #busca el tipo societario
            try:
                TipoSocietario=driver.find_element_by_id('ctl00_CPH1_UCVerCertificadoEstadoRegistralCiudadano_lblTipoSocietario').text
                print(TipoSocietario)
            except Exception:
                TipoSocietario = ''
            if(TipoSocietario=='Persona Física'):
                print('Es una PF ** ')
                obtener_datos_persona_fisica() 
                time.sleep(random.uniform(1.0,2.0))#20, 25   
                #guarda datos de PF
                
            else:
                print('Es una SOC ** ')
                obtiene_datos_socios()   
                time.sleep(random.uniform(1.0,2.0))#20, 25   
                #guarda datos de PF
            
            #vuelve
            link_volver = driver.find_element_by_id('lnkBusquedaProveedores')
            link_volver.click()   
            time.sleep(random.uniform(5.0,8.0))#20, 25               
                                   
                
    except Exception:
        print('no hay proveedores para esta convineta')
    
    #driver.find_element_by_id('ctl00_CPH1_UCBuscarProveedor_txtNumeroCUITCUIL').clear()
   
    imprime_datos_proveedor()
    imprime_datos_pf()
    imprime_datos_sociedades()
          
            

-------------------------------------------------------------------------
20047431841
nro de rows a iterar dentro de la tabla: 1
entra obtiene datos proveedor

20047431841
error RegistroPublicoComercio
error InspeccionGeneralJusticia
sale obtiene datos proveedor
entra obtiene datos clases
nro de rows a iterar dentro de la tabla Rubros: 2
sale obtiene datos clases
entra apoderado
nro de rows a iterar dentro de la tabla Apoderado: 2
sale apoderado
nro de rows a iterar dentro de la tabla Documentacion: 1
error fecha
entra antecedentes
no tiene antecedentes
entra sanciones
no tiene sanciones
Persona Física
Es una PF ** 
entra PF
20047431841
sale apoderado
-------------------------------------------------------------------------
20077944444
nro de rows a iterar dentro de la tabla: 1
entra obtiene datos proveedor
Diseño 2001
20077944444
error RegistroPublicoComercio
error InspeccionGeneralJusticia
sale obtiene datos proveedor
entra obtiene datos clases
nro de rows a iterar dentro de la tabla

nro de rows a iterar dentro de la tabla: 1
entra obtiene datos proveedor
ofi express
20208786785
error RegistroPublicoComercio
error InspeccionGeneralJusticia
sale obtiene datos proveedor
entra obtiene datos clases
nro de rows a iterar dentro de la tabla Rubros: 20
sale obtiene datos clases
entra apoderado
nro de rows a iterar dentro de la tabla Apoderado: 2
sale apoderado
nro de rows a iterar dentro de la tabla Documentacion: 6
entra antecedentes
no tiene antecedentes
entra sanciones
no tiene sanciones
Persona Física
Es una PF ** 
entra PF
20208786785
sale apoderado
-------------------------------------------------------------------------
20208931513
no hay proveedores para esta convineta
-------------------------------------------------------------------------
20214483395
nro de rows a iterar dentro de la tabla: 1
entra obtiene datos proveedor

20214483395
error RegistroPublicoComercio
error InspeccionGeneralJusticia
sale obtiene datos proveedor
entra obtiene datos clases
nro de rows

entra obtiene datos clases
nro de rows a iterar dentro de la tabla Rubros: 7
sale obtiene datos clases
entra apoderado
nro de rows a iterar dentro de la tabla Apoderado: 2
sale apoderado
nro de rows a iterar dentro de la tabla Documentacion: 6
entra antecedentes
no tiene antecedentes
entra sanciones
no tiene sanciones
Persona Física
Es una PF ** 
entra PF
20290019762
sale apoderado
-------------------------------------------------------------------------
20294014986
no hay proveedores para esta convineta
-------------------------------------------------------------------------
20294027395
no hay proveedores para esta convineta
-------------------------------------------------------------------------
20294375849
no hay proveedores para esta convineta
-------------------------------------------------------------------------
20311777387
no hay proveedores para esta convineta
-------------------------------------------------------------------------
20315559848
nro de rows a iterar dentro d

entra obtiene datos clases
nro de rows a iterar dentro de la tabla Rubros: 9
sale obtiene datos clases
entra apoderado
nro de rows a iterar dentro de la tabla Apoderado: 2
sale apoderado
nro de rows a iterar dentro de la tabla Documentacion: 6
entra antecedentes
no tiene antecedentes
entra sanciones
no tiene sanciones
Persona Física
Es una PF ** 
entra PF
23353621629
sale apoderado
-------------------------------------------------------------------------
27061984629
no hay proveedores para esta convineta
-------------------------------------------------------------------------
27102068543
nro de rows a iterar dentro de la tabla: 1
entra obtiene datos proveedor
Ramos María Cristina
27102068543
error RegistroPublicoComercio
error InspeccionGeneralJusticia
sale obtiene datos proveedor
entra obtiene datos clases
nro de rows a iterar dentro de la tabla Rubros: 3
sale obtiene datos clases
entra apoderado
nro de rows a iterar dentro de la tabla Apoderado: 2
sale apoderado
nro de rows a iterar

sale obtiene datos clases
entra apoderado
nro de rows a iterar dentro de la tabla Apoderado: 3
sale apoderado
nro de rows a iterar dentro de la tabla Documentacion: 8
entra antecedentes
no tiene antecedentes
entra sanciones
no tiene sanciones
Sociedad Anónima
Es una SOC ** 
entra socios
nro de rows a iterar dentro de la tabla socios: 2
sale socios
-------------------------------------------------------------------------
30656088636
nro de rows a iterar dentro de la tabla: 1
entra obtiene datos proveedor
MAFARAM SRL
30656088636
sale obtiene datos proveedor
entra obtiene datos clases
nro de rows a iterar dentro de la tabla Rubros: 24
sale obtiene datos clases
entra apoderado
nro de rows a iterar dentro de la tabla Apoderado: 2
sale apoderado
nro de rows a iterar dentro de la tabla Documentacion: 8
entra antecedentes
no tiene antecedentes
entra sanciones
no tiene sanciones
Sociedad Responsabilidad Limitada
Es una SOC ** 
entra socios
nro de rows a iterar dentro de la tabla socios: 3
sale 

...

no hay proveedores para esta convineta
-------------------------------------------------------------------------
30710784597
nro de rows a iterar dentro de la tabla: 1
entra obtiene datos proveedor
Bain & Company Argentina S.R.L.
30710784597
sale obtiene datos proveedor
entra obtiene datos clases
nro de rows a iterar dentro de la tabla Rubros: 3
sale obtiene datos clases
entra apoderado
nro de rows a iterar dentro de la tabla Apoderado: 3
sale apoderado
nro de rows a iterar dentro de la tabla Documentacion: 9
entra antecedentes
no tiene antecedentes
entra sanciones
no tiene sanciones
Sociedad Responsabilidad Limitada
Es una SOC ** 
entra socios
nro de rows a iterar dentro de la tabla socios: 3
sale socios
-------------------------------------------------------------------------
30710913125
nro de rows a iterar dentro de la tabla: 1
entra obtiene datos proveedor
SMARTLEDGE S.A.
30710913125
sale obtiene datos proveedor
entra obtiene datos clases
nro de rows a iterar dentro de la tabla Ru

entra sanciones
no tiene sanciones
Sociedad Responsabilidad Limitada
Es una SOC ** 
entra socios
nro de rows a iterar dentro de la tabla socios: 3
sale socios
-------------------------------------------------------------------------
30712925805
nro de rows a iterar dentro de la tabla: 1
entra obtiene datos proveedor
Qualitas Learning SRL
30712925805
sale obtiene datos proveedor
entra obtiene datos clases
nro de rows a iterar dentro de la tabla Rubros: 3
sale obtiene datos clases
entra apoderado
nro de rows a iterar dentro de la tabla Apoderado: 2
sale apoderado
nro de rows a iterar dentro de la tabla Documentacion: 7
entra antecedentes
no tiene antecedentes
entra sanciones
no tiene sanciones
Sociedad Responsabilidad Limitada
Es una SOC ** 
entra socios
nro de rows a iterar dentro de la tabla socios: 2
sale socios
-------------------------------------------------------------------------
30713649801
nro de rows a iterar dentro de la tabla: 1
entra obtiene datos proveedor
TURBO TRUENO S.A

entra sanciones
no tiene sanciones
Sociedad Responsabilidad Limitada
Es una SOC ** 
entra socios
nro de rows a iterar dentro de la tabla socios: 2
sale socios
-------------------------------------------------------------------------
30715929364
nro de rows a iterar dentro de la tabla: 1
entra obtiene datos proveedor
AUTO DANTE S.A.
30715929364
sale obtiene datos proveedor
entra obtiene datos clases
nro de rows a iterar dentro de la tabla Rubros: 4
sale obtiene datos clases
entra apoderado
nro de rows a iterar dentro de la tabla Apoderado: 3
sale apoderado
nro de rows a iterar dentro de la tabla Documentacion: 8
entra antecedentes
no tiene antecedentes
entra sanciones
no tiene sanciones
Sociedad Anónima
Es una SOC ** 
entra socios
nro de rows a iterar dentro de la tabla socios: 2
sale socios
-------------------------------------------------------------------------
30715967304
nro de rows a iterar dentro de la tabla: 1
entra obtiene datos proveedor
AUDIOLUZ SRL
30715967304
sale obtiene d

sale obtiene datos clases
entra apoderado
nro de rows a iterar dentro de la tabla Apoderado: 2
sale apoderado
nro de rows a iterar dentro de la tabla Documentacion: 9
entra antecedentes
no tiene antecedentes
entra sanciones
no tiene sanciones
Sociedad Responsabilidad Limitada
Es una SOC ** 
entra socios
nro de rows a iterar dentro de la tabla socios: 3
sale socios
-------------------------------------------------------------------------
30717752216
nro de rows a iterar dentro de la tabla: 1
entra obtiene datos proveedor
Grupo Vetrami San Martin SRL
30717752216
sale obtiene datos proveedor
entra obtiene datos clases
nro de rows a iterar dentro de la tabla Rubros: 686
sale obtiene datos clases
entra apoderado
nro de rows a iterar dentro de la tabla Apoderado: 2
sale apoderado
nro de rows a iterar dentro de la tabla Documentacion: 7
entra antecedentes
no tiene antecedentes
entra sanciones
no tiene sanciones
Sociedad Responsabilidad Limitada
Es una SOC ** 
entra socios
nro de rows a iterar

sale socios
-------------------------------------------------------------------------
214349010016
nro de rows a iterar dentro de la tabla: 1
entra obtiene datos proveedor
FARINTO S.A.
214349010016
error RegistroPublicoComercio
error InspeccionGeneralJusticia
sale obtiene datos proveedor
entra obtiene datos clases
nro de rows a iterar dentro de la tabla Rubros: 7
sale obtiene datos clases
entra apoderado
nro de rows a iterar dentro de la tabla Apoderado: 2
sale apoderado
nro de rows a iterar dentro de la tabla Documentacion: 1
error fecha
entra antecedentes
nro de rows a iterar dentro de la tabla antedecentes: 2
sale antecedentes
entra sanciones
nro de rows a iterar dentro de la tabla sanciones: 2
sale sanciones
Persona Jurídica Extranjero Sin Sucursal
Es una SOC ** 
entra socios
nro de rows a iterar dentro de la tabla socios: 1
error cuit_socio
error doc_socio
error cargo
sale socios
-------------------------------------------------------------------------
214774580014
nro de rows a i

In [50]:
driver.get(search_query)
time.sleep(random.uniform(3.0,5.0))   
driver.find_element_by_id('ctl00_CPH1_UCBuscarProveedor_txtNumeroCUITCUIL').clear()
input_cuit_proveedor = driver.find_element_by_id('ctl00_CPH1_UCBuscarProveedor_txtNumeroCUITCUIL')
    #print(input_cuit_proveedor)
input_cuit_proveedor.send_keys(cuit)
btn_cuit_proveedor=  driver.find_element_by_id('ctl00_CPH1_UCBuscarProveedor_btnBusquedaRapida') 
btn_cuit_proveedor.click()
time.sleep(random.uniform(13.0,15.0))  